# Using this package: add_numeric_missingness_indicators

```add_numeric_missingness_indicators()``` is a function you can call to add column flags that indicate whether, for a particular numeric column, there was a missing value.

## Why should you use this function?

Anyone experienced with the sklearn ecosystem knows that there are a huge range of methods for missing value imputation, and that "really they should be included within the pipeline" rather than pre-applying it to the data. This is principally because imputation on the entire dataset is a form of data leakage.

In the healthcare setting data quality varies wildly and it is critical for a machine learning practitioner to be aware of how missing values are influencing a model's performance. It is also exceediongly valuable for clinical interpretation of outputs.

To understand how a model works, a standard workflow is to pass the finalized model to the ```Dalex``` package. One limitation of this package is that imputation within a pipeline *can* obscure how missingness is being used for predictive purposes, particularly for numeric features. As such it is helpful to explicitly include missingness for each numeric column as a feature before it is passed to an sklearn pipeline.

Importantly, the value of the NULLs are not imputed in this function but left as a job for the pipeline.

## How to use this function?

To use the function, first import it

In [3]:
import pandas as pd
import numpy as np
from risk_stratifier import add_numeric_missingness_indicators, validate_binary_y_and_X

Lets generate a permissable y and X

In [15]:
y = pd.Series(np.random.randint(0, 2, size=200), dtype="int64")
# 200-row categorical column with "yes"/"no"
yes_no_col = np.random.choice(['yes', None], size=200)

# 200-row numeric column with some missing values
numeric_col = np.random.randn(200)
mask = np.random.rand(200) < 0.2   # ~20% missing
numeric_col[mask] = np.nan

# build the DataFrame
df = pd.DataFrame({
    'flag': yes_no_col,
    'value': numeric_col
})

Are these objects valid for use with this package?

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   flag    80 non-null     object 
 1   value   164 non-null    float64
dtypes: float64(1), object(1)
memory usage: 3.3+ KB


In [17]:
df.head(10)

,flag,value
0,yes,NaN
1,None,1.572045
2,yes,-1.184205
3,None,-1.078361
4,yes,1.019732
5,None,-1.686749
6,yes,0.651148
7,None,-1.834633
8,None,NaN
9,yes,2.470835


One column is a float with missing values and the other is an object (string) with missing values.

Lets check its permissable for use in this package.

In [18]:
validate_binary_y_and_X(y = y, X = df)

X and y data provided for modelling are permissable.


In [19]:
add_numeric_missingness_indicators(df)

,flag,value,value_missingness
0,yes,NaN,missing
1,None,1.572045,present
2,yes,-1.184205,present
3,None,-1.078361,present
4,yes,1.019732,present
...,...,...,...
195,None,NaN,missing
196,None,NaN,missing
197,yes,-0.902725,present
198,None,-0.481903,present


Notice that only the numeric column has had a new feature indicating missingness created, despite the string column having missing values too.